# Synthetic data generation (SDV)

Trains a Gaussian Copula synthesizer ([SDV](https://sdv.dev/)) on the tabular attributes of the original dataset (gender, coarsened lat/long, visits, age) and samples a synthetic dataset of the same size, then runs SDV's built-in diagnostic and quality checks.

**Input:** the original point dataset.
**Output:** `data/synthetic/synthetic_points.shp`

In [ ]:
# Check that the optional heavy dependencies are installed before continuing
import importlib.util

missing = [pkg for pkg in ("sdv", "ctgan") if importlib.util.find_spec(pkg) is None]
if missing:
    raise ModuleNotFoundError(
        f"Missing package(s): {', '.join(missing)}. Install with "
        "'pip install -r requirements.txt' before running this notebook."
    )


In [ ]:
# Imports
import os
from pathlib import Path

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from sdv.metadata import Metadata
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.evaluation.single_table import run_diagnostic, evaluate_quality


In [ ]:
#Paths
# Data directory. Override with the DATA_DIR environment variable to
# point at a different copy of the data without editing this file, e.g.
#   DATA_DIR=/path/to/data jupyter execute this_notebook.ipynb
# The data itself is not part of this repository -- see README.md.
DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))

OUT_DIR = DATA_DIR / "synthetic"
OUT_DIR.mkdir(parents=True, exist_ok=True)

#Loading the original dataset
odp = gpd.read_file(DATA_DIR / "original" / "2008_eov_illness_stand.shp")


In [ ]:
#read and prepare the data
if ("lat" not in odp.columns) or ("long" not in odp.columns):
    #transform to WGS84 to get lat/long
    odp_wgs84 = odp.to_crs("EPSG:4326")
    odp["long"] = odp_wgs84.geometry.x
    odp["lat"] = odp_wgs84.geometry.y

#prepare training data for SDV
train_df = odp[["GENDER", "lat", "long", "VISITS", "AGE"]].copy()

#drop identifiers, if present, before training the synthesizer
for col in ["patient_id", "name", "address"]:
    if col in train_df.columns:
        train_df = train_df.drop(columns=col)

#coarsen lat/long to reduce precision and thus increase privacy
train_df["lat"] = train_df["lat"].round(3)
train_df["long"] = train_df["long"].round(3)

In [ ]:
#generate synthetic metadata with SDV
metadata = Metadata.detect_from_dataframe(
    data=train_df,
    table_name="health_events"
)

#update metadata to specify data types and representations
metadata.update_column("GENDER", sdtype="categorical")
metadata.update_column("lat", sdtype="numerical", computer_representation="Float")
metadata.update_column("long", sdtype="numerical", computer_representation="Float")
metadata.update_column("VISITS", sdtype="numerical", computer_representation="Int64")
metadata.update_column("AGE", sdtype="numerical", computer_representation="Int64")

#train the GaussianCopulaSynthesizer with the original data and metadata
synthesizer = GaussianCopulaSynthesizer(metadata)
synthesizer.fit(train_df)

#generate synthetic data with the same number of rows as the original dataset
synthetic = synthesizer.sample(num_rows=len(train_df))

#generate the geometry column for the synthetic data
synthetic_gdf = gpd.GeoDataFrame(
    synthetic,
    geometry=[Point(xy) for xy in zip(synthetic["long"], synthetic["lat"])],
    crs="EPSG:4326"
)

#retransform the synthetic GeoDataFrame to the original CRS of the ODP dataset
synthetic_gdf_orig_crs = synthetic_gdf.to_crs(odp.crs)

#save the synthetic data to a new shapefile
out_path = OUT_DIR / "synthetic_points.shp"
synthetic_gdf_orig_crs.to_file(out_path)

#run diagnostic evaluation to compare the original and synthetic datasets
diagnostic = run_diagnostic(
    real_data=train_df,
    synthetic_data=synthetic,
    metadata=metadata
)
print(diagnostic)

#calculate the overall quality score of the synthetic data compared to the original data
quality = evaluate_quality(
    real_data=train_df,
    synthetic_data=synthetic,
    metadata=metadata,
    verbose=False,
)

diagnostic_score = diagnostic.get_score()
quality_score = quality.get_score()

print(f"Diagnostic Score: {diagnostic_score:.3f}")
print(f"Quality Score: {quality_score:.3f}")